# TerraScout AI — Producto 2: Valuación Hedónica, Métricas e Inferencia ML

Notebook estático y 100% autocontenido (funciona directo en **Google Colab** y **Jupyter local** sin dependencias externas). Utiliza únicamente el dataset limpio actual (`894` lotes reales de Córdoba).

## 1. Objetivo de negocio y definición del Target

- **Problema de negocio:** Estimar el valor justo por `m²` de cada terreno en los valles de **Calamuchita y Punilla (Córdoba)** para detectar oportunidades de arbitraje inmobiliario (`>= 20%` más baratos que su valor predicho), excluyendo lotes inviables por pendiente (`> 12°`) o riesgo ambiental (`ALTO`).
- **Target elegido — `y = log(precio_m2_usd)`:**
  - Los precios por `m²` tienen asimetría positiva marcada por terrenos chicos o ubicaciones premium.
  - Modelar el logaritmo natural estabiliza la varianza y optimiza el error relativo porcentual.
  - Al predecir, se des-transforma con `precio_modelo_m2 = exp(y_hat)` para volver a `USD/m²`.

## 2. Carga del dataset limpio actual y reglas de negocio

In [ ]:
import json
import pickle
import re
import tempfile
import unicodedata
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    VotingRegressor,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RULES = {
    'target_column': 'precio_m2_usd',
    'target_log_transform': True,
    'test_size': 0.20,
    'random_state': 42,
    'cv_folds': 5,
    'stratify_column': 'valle_corredor',
    'pendiente_alerta_grados': 12.0,
    'umbral_descuento_minimo_pct': 20.0,
    'umbral_gap_sospechoso_pct': 50.0,
    'excluir_riesgo_alto': True,
    'ocultar_pii': True,
}

DATASET_URL = 'https://drive.google.com/uc?export=download&confirm=t&id=1y8F8WmbvaXI7QFljrk7KOJk0AagZq0Ji'
LOCAL_CANDIDATES = [
    Path('data/processed/dataset_limpio.csv'),
    Path('../data/processed/dataset_limpio.csv'),
]
local_path = next((p for p in LOCAL_CANDIDATES if p.is_file()), None)
df_clean = pd.read_csv(local_path if local_path is not None else DATASET_URL)

print(f'Fuente cargada: {local_path if local_path is not None else "Google Drive público"}')
print(f'Dataset limpio oficial: {len(df_clean)} lotes x {df_clean.shape[1]} columnas')
print(f'Distribución por valle: {df_clean["valle_corredor"].value_counts().to_dict()}')
display(df_clean[['precio_usd', 'superficie_m2', 'precio_m2_usd', 'pendiente_media_grados', 'score_servicios']].describe().round(2))

## 3. Features seleccionadas y Preprocesamiento anti-leakage

| Bloque de Features | Variables seleccionadas | Qué aporta al modelo hedónico |
| :--- | :--- | :--- |
| **Físicas y servicios** | `superficie_m2`, `log_superficie`, `inv_sqrt_sup`, `score_servicios`, `tiene_agua`, `tiene_luz`, `tiene_escritura`, `tiene_arboleda`, `tiene_rio`, `tiene_servicios` | Economías de escala (a mayor m², menor USD/m²) y prima por infraestructura/escritura. |
| **Topografía y ambiente** | `pendiente_media_grados`, `ndvi_sombra_verano`, `distancia_rio_m` | Complejidad de obra, sombra natural (NDVI) y cercanía al río. |
| **Geoespacial y demanda turística** | `lat_eff`, `lon_eff`, `lat_lon_rot1/2`, `valle_corredor`, `barrio_localidad`, distancias a centroides de localidades (`dist_centroide_*`), `campings_5km`, `distancia_camping_min_km`, `nps_proxy_zona` + `KMeans` geo | Micro-localización continua en Calamuchita/Punilla, densidad de competencia y demanda turística. |
| **Texto / NLP** | 10 indicadores `kw_*` (`kw_lago_dique`, `kw_barrio_privado`, `kw_ruta_asfalto`, `kw_construido`, `kw_financiacion`, `kw_esquina`, `kw_vista`, `kw_loteo`, `kw_gas`, `kw_alambrado`), `txt_len` + `TfidfVectorizer` + `TruncatedSVD` | Distingue terrenos frente al lago, en barrios cerrados, sobre asfalto o con mejoras construidas. |

**Columnas excluidas:** `precio_usd` y `precio_m2_usd` (target, para evitar *data leakage*) y columnas PII (`telefono_en_descripcion`, `vendedor_nombre`, `url_vendedor`).

In [ ]:
def _haversine_matrix(lats1, lons1, lats2, lons2):
    p1, p2 = np.radians(lats1)[:, None], np.radians(lats2)[None, :]
    l1, l2 = np.radians(lons1)[:, None], np.radians(lons2)[None, :]
    a = np.sin((p2 - p1) / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin((l2 - l1) / 2.0) ** 2
    return 2.0 * 6371.0 * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))

def enrich_features(df_in: pd.DataFrame):
    df = df_in.copy()
    if 'id_registro' not in df.columns and 'id_publicacion' in df.columns:
        df['id_registro'] = df['id_publicacion']
    df['id_registro'] = df['id_registro'].astype(str)
    for col in ('tiene_agua', 'tiene_luz', 'tiene_escritura', 'tiene_arboleda', 'tiene_rio', 'tiene_servicios'):
        if col not in df.columns:
            df[col] = 0

    sup = pd.to_numeric(df['superficie_m2'], errors='coerce').clip(lower=1.0)
    df['log_superficie'] = np.log1p(sup)
    df['inv_sqrt_sup'] = 1.0 / np.sqrt(sup)
    lat_arr = pd.to_numeric(df['lat_eff'], errors='coerce').to_numpy(dtype=float)
    lon_arr = pd.to_numeric(df['lon_eff'], errors='coerce').to_numpy(dtype=float)
    df['lat_lon_rot1'] = lat_arr + lon_arr
    df['lat_lon_rot2'] = lat_arr - lon_arr

    # Centroides de las 16 localidades derivables directamente desde dataset_limpio
    centroids = df.groupby('barrio_localidad')[['lat_eff', 'lon_eff']].median()
    d_loc = _haversine_matrix(lat_arr, lon_arr, centroids['lat_eff'].to_numpy(), centroids['lon_eff'].to_numpy())
    dist_loc_cols = []
    for i in range(d_loc.shape[1]):
        cname = f'dist_loc_{i}'
        df[cname] = d_loc[:, i]
        dist_loc_cols.append(cname)
    df['dist_centroide_min'] = d_loc.min(axis=1)

    tit = df['titulo_publicacion'].fillna('') if 'titulo_publicacion' in df.columns else pd.Series('', index=df.index)
    txt_all = tit.str.lower()
    df['kw_lago_dique'] = txt_all.str.contains(r'\blago\b|\bdique\b|costa del lago|frente al lago', regex=True).astype(int)
    df['kw_barrio_privado'] = txt_all.str.contains(r'barrio privado|barrio cerrado|country|condominio|club de campo', regex=True).astype(int)
    df['kw_ruta_asfalto'] = txt_all.str.contains(r'sobre ruta|asfalto|asfaltada|pavimento|ruta \d+', regex=True).astype(int)
    df['kw_construido'] = txt_all.str.contains(r'casa|cabana|cabaña|construid|preventa|complejo|quincho|pileta|piscina|galpon', regex=True).astype(int)
    df['kw_financiacion'] = txt_all.str.contains(r'financiaci|cuotas|anticipo|entrega|permuta', regex=True).astype(int)
    df['kw_esquina'] = txt_all.str.contains(r'\besquina\b|doble frente', regex=True).astype(int)
    df['kw_vista'] = txt_all.str.contains(r'vista panoramica|vista a las sierras|vista al lago|vistas', regex=True).astype(int)
    df['kw_loteo'] = txt_all.str.contains(r'loteo|urbanizaci|parcela|fraccion|fracción', regex=True).astype(int)
    df['kw_gas'] = txt_all.str.contains(r'gas natural', regex=True).astype(int)
    df['kw_alambrado'] = txt_all.str.contains(r'alambrado|perimetrado|cerrado', regex=True).astype(int)
    df['txt_len'] = np.log1p(tit.str.len())
    df['texto_full'] = tit.str.strip()

    num_features = [
        'superficie_m2', 'log_superficie', 'inv_sqrt_sup',
        'lat_eff', 'lon_eff', 'lat_lon_rot1', 'lat_lon_rot2',
        'score_servicios', 'tiene_agua', 'tiene_luz', 'tiene_escritura',
        'tiene_arboleda', 'tiene_rio', 'tiene_servicios',
        'pendiente_media_grados', 'ndvi_sombra_verano', 'distancia_rio_m',
        'campings_5km', 'distancia_camping_min_km', 'nps_proxy_zona', 'dist_centroide_min',
        'kw_lago_dique', 'kw_barrio_privado', 'kw_ruta_asfalto', 'kw_construido',
        'kw_financiacion', 'kw_esquina', 'kw_vista', 'kw_loteo', 'kw_gas', 'kw_alambrado', 'txt_len',
    ] + dist_loc_cols
    cat_features = ['valle_corredor', 'barrio_localidad']
    text_col = 'texto_full'
    return df, num_features, cat_features, text_col

df_feat, num_features, cat_features, text_col = enrich_features(df_clean)
feature_cols = num_features + cat_features + [text_col]
print(f'Features numéricas: {len(num_features)} | Categóricas: {cat_features} | Texto: {text_col} | Total X: {len(feature_cols)}')

## 4. Entrenamiento anti-leakage, comparación de 7 modelos y Métricas ML

Se separa `80% train` / `20% test` estratificado por `valle_corredor` antes de ajustar transformadores, se comparan **7 modelos** con **5-Fold Cross Validation** sobre `train`, se verifica en `test` no visto y se valida la serialización Pickle (`dif_max < 1e-6`).

In [ ]:
df_model = df_feat.dropna(subset=[RULES['target_column']]).copy()
X = df_model[feature_cols].copy()
y = np.log(df_model[RULES['target_column']].to_numpy(dtype=float))

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=RULES['test_size'],
    random_state=RULES['random_state'],
    stratify=df_model[RULES['stratify_column']],
)

n_clusters = min(30, max(5, len(X_train) // 25))
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_features),
    ('geo_cl', Pipeline([('imp', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('km', KMeans(n_clusters=n_clusters, n_init=10, random_state=RULES['random_state']))]), ['lat_eff', 'lon_eff']),
    ('txt', Pipeline([('tf', TfidfVectorizer(max_features=400, ngram_range=(1, 2), min_df=3)), ('svd', TruncatedSVD(n_components=20, random_state=RULES['random_state']))]), text_col),
])

rs = RULES['random_state']
et_est = ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2, max_features=0.5, random_state=rs)
gbr_est = GradientBoostingRegressor(n_estimators=250, max_depth=3, learning_rate=0.04, subsample=0.8, random_state=rs)
hgbr_est = HistGradientBoostingRegressor(max_depth=5, min_samples_leaf=10, learning_rate=0.04, max_iter=250, random_state=rs)

candidates = {
    'Baseline (Dummy mediana)': DummyRegressor(strategy='median'),
    'Ridge Linear': Ridge(alpha=10.0),
    'Random Forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=2, max_features=0.5, random_state=rs),
    'ExtraTrees': et_est,
    'GradientBoosting': gbr_est,
    'HistGradientBoosting': hgbr_est,
    'Ensemble (2ET + GBR + HGBR)': VotingRegressor([('et', clone(et_est)), ('gbr', clone(gbr_est)), ('hgbr', clone(hgbr_est))], weights=[2, 1, 1]),
}

cv = KFold(n_splits=RULES['cv_folds'], shuffle=True, random_state=rs)
comparison, trained_pipes = [], {}
real_test_usd = np.exp(y_test)

for name, est in candidates.items():
    pipe = Pipeline([('prep', clone(preprocessor)), ('model', est)])
    cv_r2 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='r2')
    pipe.fit(X_train, y_train)
    trained_pipes[name] = pipe
    pred_test_raw = pipe.predict(X_test)
    pred_test_usd = np.exp(pred_test_raw)
    comparison.append({
        'modelo': name,
        'r2_cv_mean': round(float(cv_r2.mean()), 4),
        'r2_cv_std': round(float(cv_r2.std()), 4),
        'r2_test_log': round(float(r2_score(y_test, pred_test_raw)), 4),
        'mae_test_usd_m2': round(float(mean_absolute_error(real_test_usd, pred_test_usd)), 2),
        'rmse_test_usd_m2': round(float(np.sqrt(mean_squared_error(real_test_usd, pred_test_usd))), 2),
    })

df_comp = pd.DataFrame(comparison).sort_values(['r2_cv_mean', 'mae_test_usd_m2'], ascending=[False, True]).reset_index(drop=True)
display(df_comp)

winner_row = df_comp.iloc[0]
dummy_row = df_comp.loc[df_comp['modelo'].str.contains('Dummy')].iloc[0]
mae_gain = (dummy_row['mae_test_usd_m2'] - winner_row['mae_test_usd_m2']) / dummy_row['mae_test_usd_m2'] * 100.0
winner_name = winner_row['modelo']
winner_pipe = trained_pipes[winner_name]

# Predicciones honestas OOF (train) + out-of-sample (test)
oof_train_m2 = np.exp(cross_val_predict(clone(winner_pipe), X_train, y_train, cv=cv))
pred_test_m2 = np.exp(winner_pipe.predict(X_test))
pred_oof_test = pd.Series(index=df_model.index, dtype=float)
pred_oof_test.loc[X_train.index] = oof_train_m2
pred_oof_test.loc[X_test.index] = pred_test_m2
split_series = pd.Series(index=df_model.index, dtype=object)
split_series.loc[X_train.index] = 'train_oof'
split_series.loc[X_test.index] = 'test'

# Entrenamiento final sobre 100% de X para producción + check Pickle
prod_pipe = clone(winner_pipe).fit(X, y)
pkl_bytes = pickle.dumps({'pipeline': prod_pipe, 'modelo_nombre': winner_name, 'feature_columns': feature_cols, 'target_log_transformed': True, 'fecha_entrenamiento': date.today().isoformat(), 'sklearn_version': sklearn.__version__})
dif_max = float(np.max(np.abs(np.exp(prod_pipe.predict(X_test)) - np.exp(pickle.loads(pkl_bytes)['pipeline'].predict(X_test)))))

print(f"Ganador por CV: {winner_name}")
print(f"R2 CV: {winner_row['r2_cv_mean']:.4f} +/- {winner_row['r2_cv_std']:.4f} | R2 Test: {winner_row['r2_test_log']:.4f}")
print(f"MAE Test: {winner_row['mae_test_usd_m2']:.2f} USD/m2 (-{mae_gain:.1f}% vs Dummy {dummy_row['mae_test_usd_m2']:.2f}) | RMSE Test: {winner_row['rmse_test_usd_m2']:.2f} USD/m2")
print(f"Verificación Pickle (dif_max): {dif_max:.2e} (< 1e-6 requerido)")

In [ ]:
prep_fit = prod_pipe.named_steps['prep']
model_fit = prod_pipe.named_steps['model']
feat_names = prep_fit.get_feature_names_out()
if hasattr(model_fit, 'feature_importances_'):
    raw_imp = np.asarray(model_fit.feature_importances_, dtype=float)
elif hasattr(model_fit, 'estimators_'):
    raw_imp = np.mean([np.asarray(e.feature_importances_, dtype=float) for e in model_fit.estimators_ if hasattr(e, 'feature_importances_')], axis=0)
else:
    raw_imp = np.zeros(len(feat_names))
imp_df = pd.DataFrame({'feature': feat_names, 'importancia_pct': raw_imp / (raw_imp.sum() or 1.0) * 100.0}).sort_values('importancia_pct', ascending=False).head(12)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
imp_df.sort_values('importancia_pct').plot(kind='barh', x='feature', y='importancia_pct', legend=False, ax=axes[0], color='#2a9d8f')
axes[0].set_title('Top 12 Importancias de Variables (%)')
axes[0].set_xlabel('Importancia (%)')

for sp_name, color in [('train_oof', '#264653'), ('test', '#e76f51')]:
    mask = split_series == sp_name
    axes[1].scatter(df_model.loc[mask, 'precio_m2_usd'], pred_oof_test.loc[mask], alpha=0.5, s=18, label=sp_name, color=color)
lim = max(df_model['precio_m2_usd'].max(), pred_oof_test.max())
axes[1].plot([0, lim], [0, lim], '--', color='#555555', linewidth=1.2, label='Identidad y=x')
axes[1].set_title('Precio Real vs. Predicho (USD/m2)')
axes[1].set_xlabel('Precio publicado real (USD/m2)')
axes[1].set_ylabel('Precio hedónico modelo (USD/m2)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Métricas de negocio y detección de arbitraje (OOF + Test)

Reglas de oportunidad:
- `gap_arbitraje_pct >= 20%`
- `pendiente_media_grados <= 12°`
- `riesgo_ambiental != 'ALTO'` (0% de lotes inviables en alertas)
- `flag_revision = 'REVISAR'` si el gap es `>= 50%` (posible error de aviso a auditar).

In [ ]:
def apply_arbitrage_rules(df_in: pd.DataFrame, pred_m2: np.ndarray) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df_in.copy()
    out['precio_modelo_m2'] = np.round(pred_m2, 2)
    out['precio_modelo_total_usd'] = np.round(out['precio_modelo_m2'] * out['superficie_m2'].astype(float), 0)
    out['gap_arbitraje_pct'] = np.where(
        out['precio_modelo_m2'] > 0,
        ((out['precio_modelo_m2'] - out['precio_m2_usd'].astype(float)) / out['precio_modelo_m2'] * 100.0).round(2),
        0.0,
    )
    cond_riesgo = (out['riesgo_ambiental'] != 'ALTO') if RULES['excluir_riesgo_alto'] else True
    out['es_oportunidad_ml'] = (
        (out['gap_arbitraje_pct'] >= RULES['umbral_descuento_minimo_pct'])
        & (out['pendiente_media_grados'].astype(float) <= RULES['pendiente_alerta_grados'])
        & cond_riesgo
    )
    out['flag_revision'] = np.where(out['gap_arbitraje_pct'] >= RULES['umbral_gap_sospechoso_pct'], 'REVISAR', 'OK')
    drop_cols = [c for c in out.columns if c.startswith('dist_loc_') or c in ('texto_full', 'telefono_en_descripcion', 'vendedor_nombre', 'url_vendedor')]
    out = out.drop(columns=drop_cols, errors='ignore')
    oport = out[out['es_oportunidad_ml']].sort_values('gap_arbitraje_pct', ascending=False).reset_index(drop=True)
    return out, oport

df_scored_eval, df_oport_eval = apply_arbitrage_rules(df_model, pred_oof_test.to_numpy())
df_scored_eval['split_evaluacion'] = split_series.to_numpy()
df_oport_eval = df_scored_eval[df_scored_eval['es_oportunidad_ml']].sort_values('gap_arbitraje_pct', ascending=False).reset_index(drop=True)

n_rev = int((df_oport_eval['flag_revision'] == 'REVISAR').sum())
n_ok = int((df_oport_eval['flag_revision'] == 'OK').sum())
print(f'Lotes evaluados (OOF + Test): {len(df_scored_eval)} | Oportunidades ML: {len(df_oport_eval)} ({n_ok} OK | {n_rev} REVISAR)')
print(f'Lotes con riesgo ALTO en oportunidades: {int((df_oport_eval["riesgo_ambiental"] == "ALTO").sum())}')

cols_vista = ['id_registro', 'barrio_localidad', 'precio_usd', 'superficie_m2', 'precio_m2_usd', 'precio_modelo_m2', 'gap_arbitraje_pct', 'pendiente_media_grados', 'riesgo_ambiental', 'flag_revision', 'split_evaluacion']
display(df_oport_eval[cols_vista].head(15))

## 6. Inferencia pura desde el modelo serializado (`.pkl`)

Demostración idéntica a la ejecución en producción (sin reentrenamiento): se guarda y recarga el artefacto Pickle desde disco y se ejecuta inferencia sobre los lotes limpios.

In [ ]:
tmp_pkl = Path(tempfile.gettempdir()) / 'terrascout_pipeline_demo.pkl'
tmp_pkl.write_bytes(pkl_bytes)

with open(tmp_pkl, 'rb') as f:
    loaded_bundle = pickle.load(f)

pred_prod_m2 = np.exp(loaded_bundle['pipeline'].predict(df_feat[loaded_bundle['feature_columns']]))
df_inf_all, df_inf_oport = apply_arbitrage_rules(df_feat, pred_prod_m2)

print('Metadatos modelo cargado (.pkl):', {
    'modelo_nombre': loaded_bundle['modelo_nombre'],
    'fecha_entrenamiento': loaded_bundle['fecha_entrenamiento'],
    'sklearn_version': loaded_bundle['sklearn_version'],
    'lotes_evaluados': len(df_inf_all),
    'oportunidades_inferencia': len(df_inf_oport),
})
cols_inf = ['id_registro', 'barrio_localidad', 'precio_m2_usd', 'precio_modelo_m2', 'precio_modelo_total_usd', 'gap_arbitraje_pct', 'es_oportunidad_ml', 'flag_revision']
display(df_inf_oport[cols_inf].head(10))